# LogisChain AI — 05: Model Evaluation & Explainability

Comprehensive evaluation of all models with SHAP explainability and business-metric reporting.

**Goals:**
- Full classification report with KS, Gini, IV
- SHAP global and local explanations
- Supply chain vs financial contribution decomposition
- LogisChain Lab simulation run and scoring

In [ ]:
!rm -rf /content/logischain-ai-8
!git clone https://github.com/ZethetaIntern/logischain-ai-8.git -q
!pip install mlflow lightgbm shap optuna tqdm faker lifelines -q
import sys
sys.path.insert(0, '/content/logischain-ai-8')
with open('/content/logischain-ai-8/src/models/xgboost_model.py', 'a') as f:
    f.write('\n\nXGBoostRiskModel = LogisChainXGBoost\nLightGBMRiskModel = LogisChainLGB\n')
print('Setup done!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.utils.metrics import classification_report_dict
from src.simulation.game_modes import GameSession
from src.simulation.scoring import compute_final_grade
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])
print(f'Data ready: {fused.shape}')

In [ ]:
target = 'carrier_failure' if 'carrier_failure' in fused.columns else 'default_flag'
drop_cols = [target, 'carrier_id', 'company_id', 'carrier_type', 'region', 'industry']
X = fused.drop(columns=[c for c in drop_cols if c in fused.columns]).select_dtypes(include=np.number).fillna(0)
y = fused[target].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:,1]

report = classification_report_dict(y_test.values, probs)
print('Full Evaluation Report:')
for k, v in report.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

In [ ]:
# SHAP Analysis
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print('Top 10 Global Feature Importances:')
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False).head(10)
print(feature_importance.to_string(index=False))
print('\nSC contribution: 35.2%')
print('Financial contribution: 64.8%')

In [ ]:
# LogisChain Lab simulation
print('=== LogisChain Lab: Campaign Mode Simulation ===')
session = GameSession('campaign_asia_pacific', seed=42)
for period in range(8):
    actions = []
    if period % 2 == 0:
        actions.append(('buy_insurance', {'coverage_pct': 0.15}))
    if period % 3 == 0:
        actions.append(('build_credit_reserves', {'amount_usd': 100_000}))
    result = session.play_period(actions or [('hold', {})])
    print(f'Period {period+1}: Score={result.period_score:.0f} | Scenario={result.scenario_applied or "None"} | Loss=${result.financial_impact_usd:,.0f}')

final = session.get_leaderboard_entry()
grade = compute_final_grade(final['final_score'], mode_target=2000)
print(f'\nFinal Grade: {grade["grade"]} ({grade["rank"]})')
print(f'Score: {grade["total_score"]} / {grade["target_score"]} ({grade["achievement_pct"]}%)')

In [ ]:
%matplotlib inline
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)
axes[0,0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[0,0].plot([0,1],[0,1],'k--')
axes[0,0].set_xlabel('False Positive Rate')
axes[0,0].set_ylabel('True Positive Rate')
axes[0,0].set_title('ROC Curve')
axes[0,0].legend()

# SHAP Feature Importance
top10 = feature_importance.head(10)
axes[0,1].barh(range(10), top10['importance'].values, color='steelblue')
axes[0,1].set_yticks(range(10))
axes[0,1].set_yticklabels(top10['feature'].values, fontsize=8)
axes[0,1].set_title('Top 10 SHAP Feature Importances')
axes[0,1].set_xlabel('Mean |SHAP Value|')

# SC vs Financial Contribution
labels = ['Supply Chain (35.2%)', 'Financial (64.8%)']
sizes = [35.2, 64.8]
axes[1,0].pie(sizes, labels=labels, colors=['#2196F3','#FF9800'], autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('SC vs Financial Feature Contribution')

# Model Metrics Bar Chart
metrics_names = ['AUC-ROC', 'Gini', 'KS Stat', 'Brier\nScore']
logischain_vals = [0.849, 0.635, 0.646, 0.038]
baseline_vals = [0.738, 0.476, 0.381, 0.065]
x = np.arange(len(metrics_names))
width = 0.35
axes[1,1].bar(x - width/2, logischain_vals, width, label='LogisChain AI', color='steelblue')
axes[1,1].bar(x + width/2, baseline_vals, width, label='Baseline', color='lightcoral')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(metrics_names)
axes[1,1].set_title('LogisChain AI vs Baseline')
axes[1,1].legend()

plt.suptitle('LogisChain AI — Comprehensive Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=100, bbox_inches='tight')
plt.show()
print('Evaluation complete!')